# NaturaLens polar bear training (R2 snapshot)

1. Runtime → Change runtime type → **GPU (T4)**
2. Add Colab Secrets: `R2_ACCOUNT_ID`, `R2_ACCESS_KEY_ID`, `R2_SECRET_ACCESS_KEY`, `R2_BUCKET`, `LABELER_URL`, `TRAIN_TOKEN`
3. Set `RUN_ID` below to the id from the labeler Runs page
4. Runtime → Run all

In [ ]:
RUN_ID = "REPLACE_WITH_RUN_ID"  # from labeler Runs page
EPOCHS = 30
BATCH_SIZE = 8
REPO_URL = "https://github.com/abhay-cs/naturalens.git"
BRANCH = "main"


In [ ]:
!pip -q install boto3 pillow requests pycocotools onnx onnxscript
!pip -q install torch torchvision --index-url https://download.pytorch.org/whl/cu124

In [ ]:
import json, os, sys, traceback, subprocess
from pathlib import Path
from datetime import datetime, timezone

import boto3
import requests
from google.colab import userdata

def secret(name, default=None):
    try:
        return userdata.get(name)
    except Exception:
        return os.environ.get(name, default)

R2_ACCOUNT_ID = secret("R2_ACCOUNT_ID")
R2_ACCESS_KEY_ID = secret("R2_ACCESS_KEY_ID")
R2_SECRET_ACCESS_KEY = secret("R2_SECRET_ACCESS_KEY")
R2_BUCKET = secret("R2_BUCKET", "naturalens-data")
LABELER_URL = (secret("LABELER_URL") or "").rstrip("/")
TRAIN_TOKEN = secret("TRAIN_TOKEN")

assert RUN_ID and RUN_ID != "REPLACE_WITH_RUN_ID", "Set RUN_ID"
assert R2_ACCOUNT_ID and R2_ACCESS_KEY_ID and R2_SECRET_ACCESS_KEY, "Missing R2 secrets"

s3 = boto3.client(
    "s3",
    endpoint_url=f"https://{R2_ACCOUNT_ID}.r2.cloudflarestorage.com",
    aws_access_key_id=R2_ACCESS_KEY_ID,
    aws_secret_access_key=R2_SECRET_ACCESS_KEY,
    region_name="auto",
)

WORK = Path("/content/naturalens_run")
WORK.mkdir(parents=True, exist_ok=True)
DATA = WORK / "data"
IMAGES = DATA / "images"
LABELS = DATA / "labels"
IMAGES.mkdir(parents=True, exist_ok=True)
LABELS.mkdir(parents=True, exist_ok=True)

def now():
    return datetime.now(timezone.utc).isoformat()

def put_json(key, obj):
    body = json.dumps(obj, indent=2).encode("utf-8")
    s3.put_object(Bucket=R2_BUCKET, Key=key, Body=body, ContentType="application/json")

def put_file(key, path, content_type="application/octet-stream"):
    s3.upload_file(str(path), R2_BUCKET, key, ExtraArgs={"ContentType": content_type})

def callback(status, **extra):
    payload = {"status": status, "updated_at": now(), **extra}
    put_json(f"runs/{RUN_ID}/status.json", {"run_id": RUN_ID, **payload})
    if LABELER_URL and TRAIN_TOKEN:
        try:
            requests.post(
                f"{LABELER_URL}/api/runs/{RUN_ID}/status",
                headers={"Authorization": f"Bearer {TRAIN_TOKEN}"},
                json=payload,
                timeout=30,
            )
        except Exception as exc:
            print("callback failed:", exc)

print("ready", RUN_ID)

In [ ]:
# Reserve R2 free-tier budget before touching the bucket.
# Caps live in the labeler Worker (80% of free tier). Colab must report its own S3 ops.
def report_quota(class_a=0, class_b=0, storage_bytes_delta=0):
    if not (LABELER_URL and TRAIN_TOKEN):
        print('skip quota report (no LABELER_URL/TRAIN_TOKEN)')
        return None
    res = requests.post(
        f"{LABELER_URL}/api/quota/report",
        headers={"Authorization": f"Bearer {TRAIN_TOKEN}"},
        json={"class_a": class_a, "class_b": class_b, "storage_bytes_delta": storage_bytes_delta},
        timeout=30,
    )
    data = res.json()
    if res.status_code == 429:
        raise RuntimeError(data.get('error') or 'R2 quota exceeded')
    res.raise_for_status()
    return data.get('quota')

q = None
if LABELER_URL and TRAIN_TOKEN:
    q = requests.get(f"{LABELER_URL}/api/quota", timeout=30).json()
    print('quota', q.get('remaining'), 'blocked', q.get('blocked'))
    if q.get('blocked', {}).get('any'):
        raise RuntimeError('R2 free-tier cap reached in labeler — aborting before downloads')
    if q.get('remaining', {}).get('class_b', 0) < 200 or q.get('remaining', {}).get('class_a', 0) < 50:
        raise RuntimeError('Not enough R2 quota headroom for this training run')


In [ ]:
callback("running", started_at=now(), epoch=0)
n_images = None  # filled after manifest load

try:
    manifest = json.loads(
        s3.get_object(Bucket=R2_BUCKET, Key=f"runs/{RUN_ID}/manifest.json")["Body"].read()
    )
    train, val = [], []
    for item in manifest["images"]:
        file_name = item["file"]
        key = item.get("image_key") or f"images/{item['id']}.jpg"
        dest = IMAGES / file_name
        if not dest.exists():
            dest.write_bytes(s3.get_object(Bucket=R2_BUCKET, Key=key)["Body"].read())
        lines = []
        for box in item.get("boxes") or []:
            lines.append(
                f"{int(box.get('cls', 0))} {box['cx']:.6f} {box['cy']:.6f} {box['w']:.6f} {box['h']:.6f}"
            )
        (LABELS / f"{Path(file_name).stem}.txt").write_text(("\n".join(lines) + "\n") if lines else "")
        (train if item.get("split") != "val" else val).append(file_name)
    (DATA / "split.json").write_text(json.dumps({"seed": 13, "train": train, "val": val}, indent=2))
    print(f"snapshot: {len(train)} train / {len(val)} val")
    # Class B: 1 GET manifest (already done outside meter) + 1 GET per image via S3
    report_quota(class_a=1, class_b=len(manifest["images"]) + 1)

    repo = WORK / "naturalens"
    if not repo.exists():
        subprocess.check_call(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(repo)])

    # Point training scripts at this snapshot.
    link = repo / "models" / "data"
    if link.exists() or link.is_symlink():
        if link.is_symlink() or link.is_file():
            link.unlink()
        else:
            import shutil
            shutil.rmtree(link)
    link.symlink_to(DATA)

    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    subprocess.check_call([sys.executable, "models/training/prepare_coco.py"], cwd=repo, env=env)
    subprocess.check_call(
        [
            sys.executable,
            "models/training/train_polar_bear.py",
            "--epochs",
            str(EPOCHS),
            "--batch-size",
            str(BATCH_SIZE),
        ],
        cwd=repo,
        env=env,
    )

    weights = repo / "models" / "weights"
    metrics_path = repo / "models" / "evaluation" / "results" / "polar_bear.json"
    metrics = json.loads(metrics_path.read_text()) if metrics_path.exists() else {}

    preds = {
        "run_id": RUN_ID,
        "images": metrics.get("images") or metrics.get("per_image") or [],
        "summary": {k: metrics.get(k) for k in ("map50", "precision", "recall", "best_epoch") if k in metrics},
    }
    # Attach image ids from the manifest when predictions only have file names.
    # Prefer training-script box arrays (gt / preds); fall back to manifest labels for gt.
    by_file = {im["file"]: im for im in manifest["images"]}
    for entry in preds["images"]:
        meta = by_file.get(entry.get("file") or "")
        if meta and "id" not in entry:
            entry["id"] = meta["id"]
        # Older metrics only stored counts in gt/pred — replace from manifest boxes if needed.
        if meta and not isinstance(entry.get("gt"), list):
            entry["gt"] = list(meta.get("boxes") or [])
        if not isinstance(entry.get("preds"), list):
            entry["preds"] = []

    # Reserve writes for metrics/preds/status + up to 3 weight objects
    report_quota(class_a=8, storage_bytes_delta=80_000_000)
    put_json(f"runs/{RUN_ID}/metrics.json", metrics)
    put_json(f"runs/{RUN_ID}/preds.json", preds)

    uploads = [
        ("polar_bear_ssdlite.pt", "best.pt", "application/octet-stream"),
        ("polar_bear_ssdlite.onnx", "polar_bear.onnx", "application/octet-stream"),
        ("polar_bear_edlite0.tflite", "polar_bear_fp16.tflite", "application/octet-stream"),
    ]
    for src_name, dest_name, ctype in uploads:
        path = weights / src_name
        if path.exists():
            key = f"runs/{RUN_ID}/{dest_name}"
            put_file(key, path, ctype)
            print("uploaded", key)

    callback(
        "done",
        finished_at=now(),
        map50=metrics.get("map50"),
        precision=metrics.get("precision"),
        recall=metrics.get("recall"),
        epoch=metrics.get("best_epoch") or EPOCHS,
    )
    print("done", metrics.get("map50"), metrics.get("precision"), metrics.get("recall"))
except Exception:
    err = traceback.format_exc()
    print(err)
    callback("failed", finished_at=now(), error=err[-4000:])
    raise
